In [27]:
import pandas as pd
import json
import os
import ast
from sentence_transformers import SentenceTransformer

# I. EMBEDDING TASKS

## 1. Getting and Embedding the tasks for every PSOC code

### a) Create a mapping where for each PSOC code, get the list of tasks

In the PSOC Excel file found in https://psa.gov.ph/classification/psoc, the **OCCUPATIONAL TITLES AND DEFINITIONS** field follows a consistent structure containing the occupation title, a brief description, a list of tasks, and additional occupation names.

Since the tasks are the primary information of interest, we extract them using the following pattern:

* Tasks begin after **"Their tasks include:"**
* Additional occupation names begin after **"Examples of the occupations classified here:"**

Thus, the text between these two markers is extracted as the occupation's task description.


In [2]:
filepath = '../data/labor_codes/2022-Updates-to-the-2012-PSOC.xlsx'
psoc_sheets = pd.read_excel(filepath, sheet_name=None, dtype={'UNIT\nGROUP':'str'})
psoc_df = pd.concat(psoc_sheets.values())
psoc_df['UNIT\nGROUP'] = psoc_df['UNIT\nGROUP'].ffill()

# Get only the code and the statements
psoc_code_statements = psoc_df[['UNIT\nGROUP', 'OCCUPATIONAL TITLES AND DEFINITIONS']].copy().dropna()
psoc_code_statements.columns = ['Code', 'Statements']
psoc_code_statements.Statements = psoc_code_statements.Statements.str.strip()

# Use the pattern to determine if it is a task or not
pattern = (
    r'(Their tasks include:|'
    r'Their tasks would include:|'
    r'In such cases, tasks would include:|'
    r'Examples of the occupations classified here:|'
    r'Some related occupations classified elsewhere:)'
)
psoc_code_statements['Type'] = (
    psoc_code_statements['Statements']
    .str.extract(pattern, expand=False)
    .map({
        'Their tasks include:': 'Task',
        'Their tasks would include:': 'Task',
        'In such cases, tasks would include:': 'Task',
        'Examples of the occupations classified here:': 'Occupations',
        'Some related occupations classified elsewhere:': 'Occupations'
    })
)
psoc_code_statements.Type = psoc_code_statements.Type.ffill()

psoc_code_statements.dropna(inplace=True)

# get the actual tasks
psoc_tasks_df = psoc_code_statements[psoc_code_statements.Type == 'Task'].copy()
# remove the pesky 'Their tasks include:'
psoc_tasks_df['Is Actual Task'] = (
    psoc_tasks_df['Statements']
    .str.contains(r'\w\)')
)
# remove the pesky a) or b) or c)
psoc_tasks_df = psoc_tasks_df[psoc_tasks_df['Is Actual Task']]
psoc_tasks_df.Statements = (
    psoc_tasks_df.Statements
    .str.extract(r'\w\)\s(.*)', expand=False)
    .str.strip()
)
psoc_tasks_df = psoc_tasks_df[['Code', 'Statements']].dropna().reset_index(drop=True)

The following PSOC Codes (3435, 8189, 7319, 3139) do not have tasks outlined by PSA. Hence, we will just use their descriptions.

In [3]:
missing_psoc_descriptions = {
    '3435': (
        'This unit group covers artistic and cultural associate '
        'professionals not classified elsewhere in Minor Group 344, '
        'Artistic, cultural and culinary associate professionals. '
        'For instance those who assist directors or actors with '
        'staging of theatrical, motion picture, television or '
        'commercial productions are classified here.'
    ),
    '8189': (
        'This unit group includes stationary plant and machine '
        'operators not classified elsewhere in sub-major group 81, '
        'Stationary plant and machine operators. The group includes, '
        'for instance, operators of machines which make silicon '
        'chips and splice cables and ropes.'
    ),
    '7319': (
        'This unit group covers handicraft workers who perform '
        'traditional handicrafts not classified elsewhere. For '
        'instance, the group includes traditional handicraft workers '
        'in non-precious metals and stone.'
    ),
    '3139': (
        'This unit group covers process control technicians not '
        'classified elsewhere in minor group 313, Process control '
        'technicians. For instance, the group includes those who '
        'operate multiple process control equipment in manufacturing '
        'assembly lines and paper and pulp production.'
    ),
}

missing_psoc_descriptions_df = (
    pd.Series(missing_psoc_descriptions)
    .reset_index()
    .rename(columns={'index': 'Code', 0: 'Statements'})
)

complete_psoc_tasks_df = pd.concat(
    [psoc_tasks_df, missing_psoc_descriptions_df]
).reset_index(drop=True)

### b) For each task, embed it into a numerical vector space

In [4]:
def embed_strings(
    df: pd.DataFrame,
    str_col: str
) -> pd.DataFrame:
    """
    Given a dataframe and a column filled with strings,
    return the full dataframe with a new column containing
    embeddings of the string column.
    """
    embedding_model = SentenceTransformer("all-mpnet-base-v2")
    embedded_col_name = str_col + " Embedded"

    df[embedded_col_name] = df[str_col].apply(
        lambda string_statement: embedding_model.encode(
            string_statement,
            normalize_embeddings=True
        ).tolist()
    )

    return df

In [5]:
embedded_psoc_tasks_df = embed_strings(complete_psoc_tasks_df, 'Statements')
embedded_psoc_tasks_df.to_csv('../data/auxiliary/embedded_psoc_tasks.csv', index=False)

## 2. Getting and Embedding the tasks for every 2019 SOC-O*NET code

In [6]:
# open the soc onet data and clean up the O*NET-SOC Code
soc_tasks_df = pd.read_excel(
    '../data/labor_codes/Task Statements.xlsx', 
    dtype={'O*NET-SOC Code':'str'}
)
soc_tasks_df = soc_tasks_df[['O*NET-SOC Code', 'Task']].copy()

In [ ]:
# embed each task into a vector space and save it
embedded_soc_tasks_df = embed_strings(soc_tasks_df, 'Task')

For SOC codes with no available task data, neighboring occupations are used as a fallback. Specifically, occupations within the same SOC group are identified based on the shared first four digits of the SOC code, and their task data is used to represent the missing occupation. This allows the analysis to retain task-level information while maintaining occupational similarity as closely as possible.

In [57]:
mca_df = pd.read_csv('../data/auxiliary/mca_soc_codes.csv',)
mca_df['2019 SOC Codes'] = mca_df['2019 SOC Codes'].apply(ast.literal_eval)

In [100]:
to_look = set(embedded_soc_tasks_df["O*NET-SOC Code"])

problem_codes = set()

for codes in mca_df["2019 SOC Codes"]:
    for code in codes:
        if code not in to_look:
            problem_codes.add(code)

# Match problem codes using the first 5 SOC digits
first_n = 5
problem_prefixes = {code[:first_n] for code in problem_codes}

fallback_df = embedded_soc_tasks_df[
    embedded_soc_tasks_df["O*NET-SOC Code"].str[:first_n].isin(problem_prefixes)
]

problematic_embedded_soc_tasks_map = {'O*NET-SOC Code':[],'Task':[], 'Task Embedded':[]}

for problem_code in problem_codes:

    prefix = problem_code[:first_n]

    matching_rows = embedded_soc_tasks_df[
        embedded_soc_tasks_df["O*NET-SOC Code"].str.startswith(prefix)
    ]

    for _, row in matching_rows.iterrows():
        problematic_embedded_soc_tasks_map["O*NET-SOC Code"].append(problem_code)
        problematic_embedded_soc_tasks_map["Task"].append(row["Task"])
        problematic_embedded_soc_tasks_map["Task Embedded"].append(row["Task Embedded"])

problematic_embedded_soc_tasks_df = pd.DataFrame(problematic_embedded_soc_tasks_map)

complete_embedded_soc_tasks_df = pd.concat([embedded_soc_tasks_df, problematic_embedded_soc_tasks_df])
complete_embedded_soc_tasks_df.to_csv('../data/auxiliary/embedded_soc_tasks.csv', index=False)

In [ ]:
# this is to check that there all codes have tasks
universe_codes = set(complete_embedded_soc_tasks_df['O*NET-SOC Code'])
for codes in mca_df['2019 SOC Codes']:
    for code in codes:
        if code not in universe_codes:
            print(code)

# II. Embedding Titles for MCA and SOC

In [73]:
# Embed the job titles of MCA
embedded_mca_title_df = embed_strings(mca_df[['Job Title']].copy(), 'Job Title')
embedded_mca_title_df.to_csv('../data/auxiliary/embedded_mca_title.csv', index=False)

In [ ]:
# Embed the SOC titles
soc_title_df = pd.read_csv('../data/labor_codes/job_titles.csv')
soc_title_df = soc_title_df[['O*NET-SOC Code', 'Job Title']].copy()
embedded_soc_title_df = embed_strings(soc_title_df, 'Job Title')
embedded_soc_title_df.to_csv('../data/auxiliary/embedded_soc_title.csv', index=False)